# 04 - Statistical Arbitrage & Pairs Trading

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huuan26/polars-anta/blob/master/examples/04_statistical_arbitrage_pairs_trading.ipynb)

In this notebook, we build a **mean-reversion pairs trade** with **polars-anta**:
two cointegrated pairs are simulated, their log-price spread is scored with a rolling
z-score (`.ta.zscore()`), the pair relationship is monitored with a rolling correlation
(`.ta.rolling_corr()`), and trades are taken when the spread stretches beyond +/-2
standard deviations and closed when it reverts to the mean.

> The data is synthetic and the backtest ignores fees, borrow costs and slippage, so the results are illustrative only.

In [ ]:
# Install polars-anta (uncomment if running in Colab)
!pip install -q polars polars-anta matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import polars_anta  # Registers the .ta namespace

print(f"Polars Version: {pl.__version__}")
print(f"polars-anta Version: {polars_anta.__version__}")

## 1. Simulate Two Cointegrated Pairs

Each pair shares one common random-walk factor (the "market" both legs follow) plus a
stationary, mean-reverting spread component, so the two legs drift together but keep
pulling back toward each other - exactly the structure a pairs trade exploits.

- `BTC/ETH`: two crypto legs starting at different price levels (100 vs 58).
- `STOCK_A/STOCK_B`: two synthetic equities starting at 240 vs 95.

Swap in your own aligned price series with the same columns (`pair`, `bar`, `close_a`, `close_b`).

In [ ]:
np.random.seed(42)
n_bars = 1000


def make_pair(name: str, start_a: float, start_b: float, phi: float = 0.92) -> pl.DataFrame:
    # Common factor: a single random walk both legs share.
    factor = np.cumsum(np.random.normal(0.0003, 0.011, n_bars))
    # Stationary spread: AR(1) noise that keeps reverting to zero.
    shocks = np.random.normal(0.0, 0.012, n_bars)
    spread_noise = np.zeros(n_bars)
    for t in range(1, n_bars):
        spread_noise[t] = phi * spread_noise[t - 1] + shocks[t]

    log_a = np.log(start_a) + factor
    log_b = np.log(start_b) + factor + spread_noise
    return pl.DataFrame(
        {
            "pair": [name] * n_bars,
            "bar": np.arange(n_bars),
            "close_a": np.exp(log_a),
            "close_b": np.exp(log_b),
        }
    )


df = pl.concat(
    [
        make_pair("BTC/ETH", 100.0, 58.0),
        make_pair("STOCK_A/STOCK_B", 240.0, 95.0),
    ]
)
print(df.head())

## 2. Compute the Spread, Z-Score and Rolling Correlation with polars-anta

The spread is the log price ratio, $\text{Spread} = \log(P_A) - \log(P_B)$ (a hedge
ratio of $\beta = 1$; estimate a time-varying $\beta$ with `.ta.rolling_beta()` on log
returns when the legs trade at very different scales).

- **Z-Score (20)** via `.ta.zscore(20)`: how many standard deviations the spread sits from its recent mean.
- **Rolling correlation (60)** via `.ta.rolling_corr(...)`: confirms the pair is still moving together before trusting the spread.

In [ ]:
pair_df = df.with_columns(
    (pl.col("close_a").log() - pl.col("close_b").log()).alias("spread")
).with_columns(
    pl.col("spread").ta.zscore(20).over("pair").alias("spread_z"),
    pl.col("close_a").ta.rolling_corr(pl.col("close_b"), 60).over("pair").alias("roll_corr_60"),
)

print(
    pair_df.group_by("pair", maintain_order=True).agg(
        pl.col("roll_corr_60").mean().round(3).alias("avg_roll_corr"),
        pl.col("roll_corr_60").min().round(3).alias("min_roll_corr"),
        pl.col("spread_z").abs().max().round(2).alias("max_abs_z"),
    )
)

## 3. Generate Mean-Reversion Signals

The position is stateful - it is held until the spread reverts:

- **Long the spread** (buy A, sell B) when $Z \le -2.0$: A is cheap relative to B.
- **Short the spread** (sell A, buy B) when $Z \ge +2.0$: A is expensive relative to B.
- **Exit** when $|Z| \le 0.5$: the spread has reverted to its mean.

Entry/exit events set the position; a forward fill carries it across the bars in between.

In [ ]:
signal_df = pair_df.with_columns(
    pl.when(pl.col("spread_z") <= -2.0)
    .then(1)
    .when(pl.col("spread_z") >= 2.0)
    .then(-1)
    .when(pl.col("spread_z").abs() <= 0.5)
    .then(0)
    .otherwise(None)
    .alias("position_event")
).with_columns(
    pl.col("position_event").forward_fill().over("pair").fill_null(0).alias("position")
)

trades = signal_df.with_columns(
    (pl.col("position") != pl.col("position").shift(1, fill_value=0).over("pair")).alias("changed")
)
print(
    trades.group_by("pair", maintain_order=True).agg(
        ((pl.col("changed")) & (pl.col("position") != 0)).sum().alias("trades_opened"),
        (pl.col("position") != 0).sum().alias("bars_in_position"),
    )
)

## 4. Backtest and Performance Summary

A dollar-neutral pair position earns approximately the change in the log spread, so the
strategy return on each bar is the previous bar's position times the spread change
(the position is shifted by one bar to avoid look-ahead). The benchmark holds both legs
equal-weighted (buy & hold).

In [ ]:
bt = signal_df.with_columns(
    (pl.col("spread") - pl.col("spread").shift(1).over("pair")).fill_null(0.0).alias("spread_chg"),
    ((pl.col("close_a").pct_change() + pl.col("close_b").pct_change()) / 2)
    .over("pair")
    .fill_null(0.0)
    .alias("buy_hold_ret"),
).with_columns(
    (pl.col("position").shift(1, fill_value=0).over("pair") * pl.col("spread_chg")).alias(
        "strategy_ret"
    )
)

summary = bt.group_by("pair", maintain_order=True).agg(
    ((1 + pl.col("strategy_ret")).product() - 1).round(4).alias("strategy_return"),
    ((1 + pl.col("buy_hold_ret")).product() - 1).round(4).alias("buy_hold_return"),
)
print(summary)

equity = (
    bt.group_by("bar")
    .agg(
        pl.col("strategy_ret").mean().alias("strategy"),
        pl.col("buy_hold_ret").mean().alias("buy_hold"),
    )
    .sort("bar")
    .with_columns((pl.col("strategy", "buy_hold") + 1).cum_prod())
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(equity["bar"], equity["strategy"], label="Pairs strategy", color="#1f77b4", lw=1.5)
ax.plot(equity["bar"], equity["buy_hold"], label="Buy & hold (both legs)", color="#ff7f0e", lw=1.5)
ax.set_title("Pairs Trading: Strategy vs Buy & Hold", fontsize=13, fontweight="bold")
ax.set_xlabel("Bar")
ax.set_ylabel("Growth of 1.0")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
one_pair = signal_df.filter(pl.col("pair") == "BTC/ETH")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.plot(one_pair["bar"], one_pair["spread"], color="#1f77b4", lw=1.2)
ax1.set_title("BTC/ETH Log Spread", fontsize=12, fontweight="bold")
ax1.set_ylabel("Spread")
ax1.grid(True, alpha=0.3)

ax2.plot(one_pair["bar"], one_pair["spread_z"], color="#2ca02c", lw=1.2)
for level, style in [(2.0, "--"), (0.0, "-"), (-2.0, "--")]:
    ax2.axhline(level, color="red" if level != 0.0 else "gray", ls=style, lw=1)
ax2.set_title("Spread Z-Score with Entry (+/-2) and Exit (+/-0.5) Bands", fontsize=12, fontweight="bold")
ax2.set_xlabel("Bar")
ax2.set_ylabel("Z-Score")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Next Steps

- Estimate the hedge ratio over time with `.ta.rolling_beta()` on log returns instead of fixing $\beta = 1$, and rebuild the spread as $\log(P_A) - \beta_t \log(P_B)$.
- Gate entries on the rolling correlation (e.g. only trade while `roll_corr_60 > 0.7`) so a pair that stops moving together is left alone.
- Add realistic frictions - fees, borrow costs and slippage - before reading anything into the equity curve.